In [9]:
import pandas as pd
import string
import re
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score

import torch
from transformers import CLIPModel, CLIPTokenizer

In [8]:
PATH_CKPT_CLIP14 = '/home/gridsan/manderson/ovdsat/weights/clip-vit-large-patch14'
CLIPModel.from_pretrained(PATH_CKPT_CLIP14)

`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["bos_token_id"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["eos_token_id"]` will be overriden.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05,

In [5]:
def normalize(x):
    x = str(x).strip().lower()
    x = x.translate(str.maketrans("", "", string.punctuation))
    x = re.sub(r"\s+", " ", x)
    return x

In [6]:
# Event detection

csv_path = "/home/gridsan/manderson/skyscraper-s2/out/vqa/detection_videollava.csv"
df = pd.read_csv(csv_path)

y_true = df["ground_truth"] == 'yes'
y_pred = df["prediction"] == 'yes'

print("accuracy:", accuracy_score(y_true, y_pred))
print("precision:", precision_score(y_true, y_pred, zero_division=0))
print("recall:", recall_score(y_true, y_pred, zero_division=0))
print("f1:", f1_score(y_true, y_pred, zero_division=0))

accuracy: 0.49066467513069456
precision: 0.7201946472019465
recall: 0.3429895712630359
f1: 0.46467817896389324


In [7]:
# Event classification

csv_path = "/home/gridsan/manderson/skyscraper-s2/out/vqa/classification_videollava.csv"
df = pd.read_csv(csv_path)

y_true = df["ground_truth"].apply(normalize)
y_pred = df["prediction"].apply(normalize)

print("accuracy:", accuracy_score(y_true, y_pred))
print("precision:", precision_score(y_true, y_pred, average="weighted", zero_division=0))
print("recall:", recall_score(y_true, y_pred, average="weighted", zero_division=0))
print("f1:", f1_score(y_true, y_pred, average="weighted", zero_division=0))

#print(classification_report(y_true, y_pred, zero_division=0))

accuracy: 0.34652725914861837
precision: 0.4110787177990217
recall: 0.34652725914861837
f1: 0.2639873950304145


In [20]:
# Event description

csv_path = "/home/gridsan/manderson/skyscraper-s2/out/vqa/description_videollava.csv"
df = pd.read_csv(csv_path)

# --- CIDEr ---
refs = {}
hyps = {}

for i, row in df.iterrows():
    refs[i] = [{"caption": str(row["ground_truth"])}]
    hyps[i] = [{"caption": str(row["prediction"])}]

tokenizer = PTBTokenizer()
refs_tok = tokenizer.tokenize(refs)
hyps_tok = tokenizer.tokenize(hyps)

cider_score, _ = Cider().compute_score(refs_tok, hyps_tok)
print("CIDEr:", f"{cider_score:.4f}")
print()

# --- BLEU ---
smooth = SmoothingFunction().method1

bleu1, bleu2 = [], []

for _, row in df.iterrows():
    ref = [str(row["ground_truth"]).split()]
    hyp = str(row["prediction"]).split()

    bleu1.append(sentence_bleu(ref, hyp, weights=(1,0,0,0), smoothing_function=smooth))
    bleu2.append(sentence_bleu(ref, hyp, weights=(0.5,0.5,0,0), smoothing_function=smooth))
    
print("BLEU-1:", f"{sum(bleu1)/len(bleu1):.4f}")
print("BLEU-2:", f"{sum(bleu2)/len(bleu2):.4f}")
print()

# --- ROUGE ---
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

r1, r2, rL = [], [], []

for _, row in df.iterrows():
    scores = scorer.score(str(row["ground_truth"]), str(row["prediction"]))
    r1.append(scores["rouge1"].fmeasure)
    r2.append(scores["rouge2"].fmeasure)
    rL.append(scores["rougeL"].fmeasure)

print("ROUGE-1:", f"{sum(r1)/len(r1):.4f}")
print("ROUGE-2:", f"{sum(r2)/len(r2):.4f}")
print("ROUGE-L:", f"{sum(rL)/len(rL):.4f}")
print()

# --- BERTScore ---
preds = df["prediction"].astype(str).tolist()
gts = df["ground_truth"].astype(str).tolist()

P, R, F1 = score(preds, gts, lang="en", verbose=True)

print("BERTScore Precision:", f"{P.mean().item():.4f}")
print("BERTScore Recall:", f"{R.mean().item():.4f}")
print("BERTScore F1:", f"{F1.mean().item():.4f}")

PTBTokenizer tokenized 34197 tokens at 415245.17 tokens per second.
PTBTokenizer tokenized 27933 tokens at 383254.90 tokens per second.


CIDEr: 0.0459

BLEU-1: 0.2036
BLEU-2: 0.0633

ROUGE-1: 0.2555
ROUGE-2: 0.0318
ROUGE-L: 0.1741



Loading weights: 100%|██████████| 389/389 [00:00<00:00, 11934.46it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


100%|██████████| 27/27 [00:03<00:00,  7.62it/s]


computing greedy matching.


100%|██████████| 14/14 [00:00<00:00, 82.56it/s]

done in 3.72 seconds, 236.74 sentences/sec
BERTScore Precision: 0.8625
BERTScore Recall: 0.8573
BERTScore F1: 0.8599


In [13]:
# Event description (CLIP score)

csv_path = "/home/gridsan/manderson/skyscraper-s2/out/vqa/description_videollava.csv"
df = pd.read_csv(csv_path)

device = "cuda" if torch.cuda.is_available() else "cpu"

# load model + tokenizer
PATH_CKPT_CLIP14 = '/home/gridsan/manderson/ovdsat/weights/clip-vit-large-patch14'
model = CLIPModel.from_pretrained(PATH_CKPT_CLIP14)
tokenizer = CLIPTokenizer.from_pretrained(PATH_CKPT_CLIP14)
model = model.to(device)
model.eval()

# get text data
preds = df["prediction"].astype(str).tolist()
gts = df["ground_truth"].astype(str).tolist()

# compute similarity
scores = []

with torch.no_grad():
    for pred, gt in zip(preds, gts):
        tokens = tokenizer([pred, gt], padding=True, truncation=True, return_tensors="pt")
        tokens = {k: v.to(device) for k, v in tokens.items()}

        feats = model.get_text_features(**tokens)
        feats = feats / feats.norm(dim=-1, keepdim=True)

        sim = (feats[0] @ feats[1]).item()
        scores.append(sim)

clipscore = sum(scores) / len(scores)

print(f"CLIPScore: {clipscore:.4f}")

`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["bos_token_id"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["eos_token_id"]` will be overriden.


CLIPScore: 0.5061
